# 65 — LGBM: CRC + Single-Conc (FDR-weighted pseudo-pEC50)

Uses the 21,003-row single-concentration screen. Converts log2FC → pseudo-pEC50 calibrated against the CRC overlap. Weight = FDR-adjusted confidence × concentration correction.


In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, standardize_smiles
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED = 42
N_FOLDS = 5
LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1, random_state=SEED,
    verbose=-1, n_jobs=4,
)


In [2]:
def full_metrics(y_true, y_pred, cliff_pairs_df=None, label=""):
    """RAE, MAE, R², Pearson, Spearman, Kendall, Cliff_accuracy."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]

    mae_v  = float(np.mean(np.abs(yt - yp)))
    rae_v  = mae_v / float(np.mean(np.abs(yt - yt.mean()))) if yt.std() > 0 else float("nan")
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - yt.mean()) ** 2))
    r2_v   = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    pr_v, _ = stats.pearsonr(yt, yp)
    sp_v, _ = stats.spearmanr(yt, yp)
    kt_v, _ = stats.kendalltau(yt, yp)

    m = dict(RAE=rae_v, MAE=mae_v, R2=r2_v,
             Pearson=pr_v, Spearman=sp_v, Kendall=kt_v)

    if cliff_pairs_df is not None and len(cliff_pairs_df) > 0:
        correct = total = 0
        for _, row in cliff_pairs_df.iterrows():
            ia, ii = int(row.get("idx_active", -1)), int(row.get("idx_inactive", -1))
            if 0 <= ia < len(yp) and 0 <= ii < len(yp):
                correct += int(yp[ia] > yp[ii])
                total   += 1
        m["Cliff_acc"] = correct / total if total else float("nan")

    if label:
        cliff_str = f"  Cliff_acc={m.get('Cliff_acc', float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f}  MAE={mae_v:.4f}  R²={r2_v:.4f}  "
              f"Pearson={pr_v:.4f}  Spearman={sp_v:.4f}  Kendall={kt_v:.4f}{cliff_str}")
    return m


In [3]:
tr = load_train()
te = load_test()
print(f"CRC train: {len(tr):,}  |  Test: {len(te):,}")

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)
active_mask = y_tr >= 5.5
print(f"X_tr: {X_tr.shape}  actives: {active_mask.sum()}")

cliff_pairs = (pd.read_parquet(DATA_PROCESSED / "cliff_pairs.parquet")
               if (DATA_PROCESSED / "cliff_pairs.parquet").exists()
               else pd.DataFrame())
print(f"Cliff pairs available: {len(cliff_pairs)}")


CRC train: 4,139  |  Test: 513


X_tr: (4139, 2265)  actives: 380
Cliff pairs available: 149


In [4]:
def run_cv(X_int, y_int, splits, X_ext=None, y_ext=None, w_ext=None,
           label="", params=LGBM_PARAMS):
    oof = np.full(len(y_int), np.nan)
    for fold, (tr_idx, va_idx) in enumerate(splits):
        Xf = X_int[tr_idx]; yf = y_int[tr_idx]
        Xv = X_int[va_idx]; yv = y_int[va_idx]
        wf = np.ones(len(yf), dtype=np.float32)
        if X_ext is not None and len(X_ext) > 0:
            Xf = np.vstack([Xf, X_ext])
            yf = np.concatenate([yf, y_ext])
            wf = np.concatenate([wf, w_ext if w_ext is not None
                                  else np.ones(len(y_ext), dtype=np.float32)])
        m = lgb.train(params, lgb.Dataset(Xf, label=yf, weight=wf),
                      valid_sets=[lgb.Dataset(Xv, label=yv)],
                      callbacks=[lgb.early_stopping(50, verbose=False),
                                 lgb.log_evaluation(-1)])
        oof[va_idx] = m.predict(Xv)
        print(f"  fold {fold+1}  val_RAE={rae(yv, oof[va_idx]):.4f}", flush=True)
    m_all    = full_metrics(y_int, oof, cliff_pairs, label=label)
    m_active = full_metrics(y_int[active_mask], oof[active_mask],
                            label=f"{label} [active≥5.5]")
    return oof, m_all, m_active


def train_final_and_predict(X_tr_all, y_tr_all, w_tr_all, X_te, params=LGBM_PARAMS):
    m = lgb.train(params, lgb.Dataset(X_tr_all, label=y_tr_all, weight=w_tr_all),
                  callbacks=[lgb.log_evaluation(-1)])
    return np.clip(m.predict(X_te), y_tr_all.min() - 0.5, y_tr_all.max() + 0.5)


In [5]:
from pxr.data import load_single_conc

sp = load_single_conc()
print(f"Single-conc raw: {len(sp):,}")

# Calibrate log2FC → pEC50 via CRC overlap
import re
tr_inchikeys = set(tr["smiles"].map(
    lambda s: __import__("pxr.chem", fromlist=["to_inchikey"]).to_inchikey(s) or ""))
sp["inchikey"] = sp["smiles"].map(
    lambda s: __import__("pxr.chem", fromlist=["to_inchikey"]).to_inchikey(s) or "")
sp_overlap = sp[sp["inchikey"].isin(tr_inchikeys)].merge(
    tr[["smiles","pec50"]].assign(
        inchikey=tr["smiles"].map(
            lambda s: __import__("pxr.chem", fromlist=["to_inchikey"]).to_inchikey(s))),
    on="inchikey")

if len(sp_overlap) > 50:
    from scipy.stats import linregress
    slope, intercept, r, _, _ = linregress(
        sp_overlap["log2_fc_estimate"].clip(-6, 6),
        sp_overlap["pec50"])
    print(f"Calibration: slope={slope:.3f}  intercept={intercept:.3f}  r={r:.3f}  n={len(sp_overlap)}")
else:
    slope, intercept = 0.496, 5.10  # from nb26 empirical fit
    print(f"Using pre-fit calibration: slope={slope}  intercept={intercept}")

sp["pec50_pseudo"] = (intercept + slope * sp["log2_fc_estimate"].clip(-6, 6)).clip(3.0, 7.5)

# FDR-based weight (cap at 1.0)
if "fdr_bh" in sp.columns:
    sp["weight"] = np.clip(1.0 - sp["fdr_bh"].fillna(1.0), 0.05, 1.0)
else:
    sp["weight"] = 0.3

# Concentration correction: higher concentration → lower confidence pEC50
if "concentration_m" in sp.columns:
    conc_um = sp["concentration_m"] * 1e6
    sp["weight"] *= np.clip(1.0 / np.log10(conc_um.clip(1, 100) + 2), 0.1, 1.0)

# Remove CRC overlaps (avoid double-counting)
sp_novel = sp[~sp["inchikey"].isin(tr_inchikeys)].copy()
print(f"Novel SP compounds: {len(sp_novel):,} (excluded {len(sp)-len(sp_novel)} CRC overlaps)")

X_ext_raw = impute(combined(sp_novel["smiles"].tolist()))
X_ext = X_ext_raw
y_ext = sp_novel["pec50_pseudo"].values.astype(np.float64)
w_ext = sp_novel["weight"].values.astype(np.float32)
W_EXT = float(w_ext.mean())
print(f"External SP: shape={X_ext.shape}  mean_weight={W_EXT:.3f}")


Single-conc raw: 21,003


Calibration: slope=0.712  intercept=4.073  r=0.524  n=5719
Novel SP compounds: 15,284 (excluded 5719 CRC overlaps)


External SP: shape=(15284, 2265)  mean_weight=0.715


In [6]:
print("Running scaffold 5-fold CV...")
oof, m_all, m_active = run_cv(
    X_tr, y_tr, splits,
    X_ext=X_ext if len(X_ext) > 0 else None,
    y_ext=y_ext if len(X_ext) > 0 else None,
    w_ext=w_ext if len(X_ext) > 0 else None,
    label="CRC+SP_FDR"
)
print(f"\nAugmented with {len(X_ext):,} external rows (weight scale={W_EXT:.2f})" if len(X_ext) > 0
      else "\nNo external augmentation (data empty)")

results_df = pd.DataFrame([m_all, m_active], index=["overall", "active≥5.5"])
print("\n" + results_df.round(4).to_string())


Running scaffold 5-fold CV...


  fold 1  val_RAE=0.6474


  fold 2  val_RAE=0.6624


  fold 3  val_RAE=0.6675


  fold 4  val_RAE=0.6466


  fold 5  val_RAE=0.6744


  [CRC+SP_FDR] RAE=0.6560  MAE=0.5968  R²=0.4956  Pearson=0.7353  Spearman=0.7296  Kendall=0.5315  Cliff_acc=nan
  [CRC+SP_FDR [active≥5.5]] RAE=4.5173  MAE=0.9473  R²=-12.5312  Pearson=0.0630  Spearman=0.0605  Kendall=0.0402

Augmented with 15,284 external rows (weight scale=0.71)

               RAE     MAE       R2  Pearson  Spearman  Kendall  Cliff_acc
overall     0.6560  0.5968   0.4956   0.7353    0.7296   0.5315        NaN
active≥5.5  4.5173  0.9473 -12.5312   0.0630    0.0605   0.0402        NaN


In [7]:
# Final model on all data
w_base = np.ones(len(y_tr), dtype=np.float32)
if len(X_ext) > 0:
    X_all = np.vstack([X_tr, X_ext])
    y_all = np.concatenate([y_tr, y_ext])
    w_all = np.concatenate([w_base, w_ext])
else:
    X_all, y_all, w_all = X_tr, y_tr, w_base

te_preds = train_final_and_predict(X_all, y_all, w_all, X_te)
np.save(DATA_PROCESSED / "oof_lgbm_crc_singleconc_fdr.npy", oof)
np.save(DATA_PROCESSED / "te_oof_lgbm_crc_singleconc_fdr.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub) == 513 and sub["pEC50"].notna().all()
out = SUBMISSIONS / "65_lgbm_crc_singleconc_fdr.csv"
sub.to_csv(out, index=False)
print(f"Saved {out}")
print(f"Test preds  min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}")



Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\65_lgbm_crc_singleconc_fdr.csv
Test preds  min=3.01  median=4.69  max=5.89
